# Aula 4 — logging e debugging (exemplos práticos)

Como `logging.basicConfig` só tem efeito na primeira vez que é chamado
em um processo, os exemplos abaixo rodam cada um em um processo Python
separado (via `subprocess`), para você ver a configuração de cada um
funcionando de forma isolada e previsível.

In [ ]:
import subprocess

codigo = '''
import logging

logging.basicConfig(level=logging.INFO)

logging.info("Programa iniciado")
logging.warning("Configuração não encontrada, usando padrão")
logging.error("Falha ao conectar ao banco de dados")
'''

resultado = subprocess.run(["python3", "-c", codigo], capture_output=True, text=True)
print(resultado.stderr)  # por padrão, logging escreve no stderr

## Filtrando por nível

In [ ]:
codigo = '''
import logging

logging.basicConfig(level=logging.WARNING)

logging.info("Isso NÃO vai aparecer, nível abaixo do configurado")
logging.warning("Isso VAI aparecer")
'''

resultado = subprocess.run(["python3", "-c", codigo], capture_output=True, text=True)
print(resultado.stderr)

## Experimento guiado

Troque `level=logging.WARNING` para `level=logging.DEBUG` no código acima e adicione uma chamada `logging.debug("detalhe técnico")` -- rode de novo e veja a nova mensagem aparecer.

## `getLogger` com nome de módulo e formato customizado

In [ ]:
codigo = '''
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s: %(message)s",
)

logger = logging.getLogger("pagamentos")

def processar_pagamento(valor):
    logger.info(f"Processando pagamento de {valor}")
    if valor <= 0:
        logger.error(f"Valor inválido: {valor}")
        raise ValueError("Valor de pagamento deve ser positivo")
    logger.info("Pagamento processado com sucesso")

processar_pagamento(100)
'''

resultado = subprocess.run(["python3", "-c", codigo], capture_output=True, text=True)
print(resultado.stderr)

## `logging.exception`: registrando o traceback completo

In [ ]:
codigo = '''
import logging

logging.basicConfig(level=logging.INFO)

try:
    resultado = 10 / 0
except ZeroDivisionError:
    logging.exception("Erro ao calcular resultado")
'''

resultado = subprocess.run(["python3", "-c", codigo], capture_output=True, text=True)
print(resultado.stderr)

## Sobre `breakpoint()`/`pdb`

`breakpoint()` pausa a execução interativamente -- não faz sentido
"rodar" isso automaticamente em uma célula de notebook (ela ficaria
esperando comandos do depurador para sempre). Em vez disso, teste você
mesmo: copie a função abaixo para um arquivo `.py`, chame-a, e rode
`python3 arquivo.py` no terminal para ver o prompt `(Pdb)` aparecer.

In [ ]:
codigo_exemplo_pdb = '''def calcular_desconto(preco, percentual):
    breakpoint()
    desconto = preco * percentual / 100
    return preco - desconto

print(calcular_desconto(100, 10))
'''
print(codigo_exemplo_pdb)

## Mini-desafio resolvido

**Desafio:** logger que registra tentativas de acesso, distinguindo sucesso (`INFO`) e falha (`WARNING`).

In [ ]:
codigo = '''
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("autenticacao")

def autenticar(usuario, senha, senha_correta="python123"):
    if senha == senha_correta:
        logger.info(f"Login bem-sucedido: {usuario}")
        return True
    logger.warning(f"Tentativa de login falhou: {usuario}")
    return False

autenticar("ana", "python123")
autenticar("invasor", "senha_errada")
'''

resultado = subprocess.run(["python3", "-c", codigo], capture_output=True, text=True)
print(resultado.stderr)